# UmarTransit-1B: QLoRA Fine-Tuning on Google Colab

Fine-tune **Qwen2.5-1.5B-Instruct** on a synthetic transit domain dataset using **QLoRA + Unsloth** on a free T4 GPU.

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct` (Apache 2.0, 1.54B params)
**Dataset:** 2,971 train / 335 test instruction-response pairs covering GTFS, transit operations, route analysis
**Training:** QLoRA 4-bit, LoRA rank=16, 3 epochs
**Output:** `umarfarookm/UmarTransit-1B` on Hugging Face

> **Requirements:** Run this notebook on Google Colab with **GPU runtime** (Runtime → Change runtime type → T4 GPU).

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. Configuration

All hyperparameters in one place.

In [ ]:
# ── Model ──
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MODEL_NAME = "UmarTransit-1B"
MAX_SEQ_LENGTH = 1024

# ── LoRA ──
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0  # Must be 0 for 4-bit quantization with Unsloth
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ── Training ──
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4       # Effective batch size = 16
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
SEED = 42

# ── Output ──
OUTPUT_DIR = "./umartransit-output"
HF_REPO_ID = "umarfarookm/UmarTransit-1B"

# ── Dataset URLs (raw GitHub links) ──
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/umarfarookm/transit-foundation-model/main"
TRAIN_URL = f"{GITHUB_RAW_BASE}/datasets/synthetic/train.jsonl"
TEST_URL = f"{GITHUB_RAW_BASE}/datasets/synthetic/test.jsonl"

# ── System prompt for chat template ──
SYSTEM_PROMPT = (
    "You are UmarTransit-1B, a specialized AI assistant for public transit systems "
    "and GTFS (General Transit Feed Specification) data. You provide accurate, "
    "detailed answers about transit routes, stops, schedules, transfers, and GTFS concepts."
)

## 3. Load Base Model with 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## 4. Apply LoRA Adapters

Adding trainable LoRA layers to attention and MLP modules. Only ~1-2% of parameters are trainable.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

## 5. Load & Format Dataset

Load the JSONL files and format each pair into Qwen2.5's chat template.

In [ ]:
import json
import requests
from datasets import Dataset

def load_jsonl_from_url(url):
    """Download and parse a JSONL file from a URL."""
    response = requests.get(url)
    response.raise_for_status()
    return [json.loads(line) for line in response.text.strip().split("\n") if line.strip()]

def load_jsonl_from_file(path):
    """Load a JSONL file from local path."""
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

# Try loading from GitHub first, fall back to local upload
try:
    print("Loading train set from GitHub...")
    train_data = load_jsonl_from_url(TRAIN_URL)
    print("Loading test set from GitHub...")
    test_data = load_jsonl_from_url(TEST_URL)
except Exception as e:
    print(f"GitHub download failed: {e}")
    print("Please upload train.jsonl and test.jsonl to Colab.")
    train_data = load_jsonl_from_file("train.jsonl")
    test_data = load_jsonl_from_file("test.jsonl")

print(f"Train: {len(train_data)} pairs")
print(f"Test:  {len(test_data)} pairs")
print(f"\nSample pair:")
print(f"  Q: {train_data[0]['instruction']}")
print(f"  A: {train_data[0]['response'][:150]}...")

In [ ]:
def format_chat(example):
    """Format an instruction/response pair into Qwen2.5 chat template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_data).map(format_chat)
test_dataset = Dataset.from_list(test_data).map(format_chat)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Test dataset:  {len(test_dataset)} examples")
print(f"\nFormatted example (first 500 chars):")
print(train_dataset[0]["text"][:500])

## 6. Configure Trainer

Using SFTTrainer + SFTConfig from TRL with Unsloth optimizations.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        logging_steps=LOGGING_STEPS,
        save_strategy="no",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        seed=SEED,
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=False,
    ),
)

print(f"Training configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS} effective")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Steps per epoch: ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")
print(f"  Total steps: ~{NUM_EPOCHS * len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")

## 7. Train

This will take approximately 30-60 minutes on a T4 GPU.

In [ ]:
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/3600:.1f}h)")
print(f"  GPU memory peak: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB")

## 8. Test the Fine-Tuned Model

Run inference on sample questions to see if the model learned transit domain knowledge.

In [ ]:
FastLanguageModel.for_inference(model)

def ask_umartransit(question, max_new_tokens=256):
    """Send a question to the fine-tuned model and get a response."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

test_questions = [
    "What does route_type 3 mean in GTFS?",
    "What are the required files in a GTFS feed?",
    "How many routes does the Chicago Transit Authority operate?",
    "What transit modes does Auckland Transport operate?",
    "What is the difference between calendar.txt and calendar_dates.txt?",
    "Can GTFS times exceed 24:00:00?",
]

print("=" * 70)
print("UMARTRANSIT-1B INFERENCE TEST")
print("=" * 70)
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_umartransit(q)}")
    print("-" * 70)

## 9. Compare with Test Set (Expected vs Generated)

In [ ]:
import random
random.seed(42)

categories_seen = set()
samples = []
shuffled_test = list(test_data)
random.shuffle(shuffled_test)
for item in shuffled_test:
    cat = item.get("category", "")
    if cat not in categories_seen and len(samples) < 5:
        samples.append(item)
        categories_seen.add(cat)

print("=" * 70)
print("TEST SET COMPARISON: EXPECTED vs GENERATED")
print("=" * 70)
for item in samples:
    q = item["instruction"]
    expected = item["response"]
    generated = ask_umartransit(q)

    print(f"\n[{item['category']}] Q: {q}")
    print(f"  EXPECTED:  {expected[:200]}{'...' if len(expected) > 200 else ''}")
    print(f"  GENERATED: {generated[:200]}{'...' if len(generated) > 200 else ''}")
    print("-" * 70)

## 10. Save Model Locally

Save LoRA adapter and merged 16-bit model.

In [ ]:
# Save LoRA adapter only (small, ~50 MB)
lora_output_dir = f"{OUTPUT_DIR}/lora-adapter"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
print(f"LoRA adapter saved to: {lora_output_dir}")

# Save merged model in 16-bit (full model, ~3 GB)
merged_output_dir = f"{OUTPUT_DIR}/merged-16bit"
model.save_pretrained_merged(
    merged_output_dir,
    tokenizer,
    save_method="merged_16bit",
    maximum_memory_usage=0.5,  # Prevent OOM on Colab
)
print(f"Merged 16-bit model saved to: {merged_output_dir}")

## 11. Push to Hugging Face Hub

Upload the merged model to your Hugging Face repository.

**First time?** You'll need a write token from https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
)

print(f"\nModel pushed to: https://huggingface.co/{HF_REPO_ID}")
print("Done! UmarTransit-1B is now live on Hugging Face.")